# Analysis of Questionnaire Responses using Python and associated libraries
The questionnaire responses, which are saved in a csv file consititutes the dataset for
this analysis. The questionnaire contains both closed-ended questions such as gender, age, 
media type, Yes/No responses, Likert-scale ratings, and open-ended comments, so both 
quantitative and qualitative analysis techniques will be employed.

In [4]:
## Cell 1: Install only if needed
!pip install pandas numpy matplotlib scikit-learn wordcloud

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 550.7/550.7 kB 618.5 kB/s  0:00:01.7 kB/s eta 0:00:01


In [5]:
# Import libraries

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer

try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
except ImportError:
    WORDCLOUD_AVAILABLE = False

In [11]:
# Set file paths
INPUT_CSV = "data/questionnaire_responses.csv"
OUTPUT_DIR = Path("results/questionnaire")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
# Load CSV data
df = pd.read_csv(INPUT_CSV)

df = df.drop_duplicates()
df = df.dropna(how="all")
df.columns = df.columns.str.strip()

print("Rows:", len(df))
print("columns:", len(df.columns))
df.head()

Rows: 0
columns: 19


,Timestamp,1. What is your gender?,2. What is your highest academic qualification?,3. Which of the following age do you belong to?,3. What is you current employment status?,4. What types of media organisations have you worked for? [Select all that applies],5. Which of the following medium of communication does your organisation use for broadcasting services?,6. What is your current job title or designation in your organisation?,7. Have you at any point considered publishing articles based on academic research findings?,"8. If your response to the above question Yes, how easy was it for you to find the relevant research findings for your article?","9. What challenges, if any, have you encountered when writing your articles based on research findings? [Select all that apply]","10. If your response to the above question is Other, briefly describe challenges you faced.",11. How much support support do you get from your superiors when you want to report on academic research work?,"12. If your response to the above question is No, briefly describe what features you have have wanted to be include or challenges you faced to move from one part to another.",16. Have you published any article based on results from academic research?,"17. If you response to the above question is Yes, briefly describe the response from your supervisor, readers, and other journalist.",18. What factors influenced your choice of articles to write or publish about? [Select all that apply.],19. Are there other observations or comment you would like to make?,"20. If you response to the above question is Yes, write your comment, observation or suggestions here."


In [13]:
# Standardize column names
def normalize_column_name(col):
    col = col.lower().strip()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    return col.strip()

original_columns = df.columns.tolist()
df.columns = [normalize_column_name(col) for col in original_columns]

column_mapping = pd.DataFrame({
    "original_columns": original_columns,
    "renamed_columns": df.columns
})

column_mapping.to_csv(OUTPUT_DIR / "column_mapping.csv", index=False)
column_mapping

,original_columns,renamed_columns
0,timestamp,timestamp
1,1_what_is_your_gender_,1_what_is_your_gender_
2,2_what_is_your_highest_academic_qualification_,2_what_is_your_highest_academic_qualification_
3,3_which_of_the_following_age_do_you_belong_to_,3_which_of_the_following_age_do_you_belong_to_
4,3_what_is_you_current_employment_status_,3_what_is_you_current_employment_status_
5,4_what_types_of_media_organisations_have_you_w...,4_what_types_of_media_organisations_have_you_w...
6,5_which_of_the_following_medium_of_communicati...,5_which_of_the_following_medium_of_communicati...
7,6_what_is_your_current_job_title_or_designatio...,6_what_is_your_current_job_title_or_designatio...
8,7_have_you_at_any_point_considered_publishing_...,7_have_you_at_any_point_considered_publishing_...
9,8_if_your_response_to_the_above_question_yes_h...,8_if_your_response_to_the_above_question_yes_h...


In [ ]:
# Helper functions
def save_frequency_table(df, column):
    freq = df[column].value_counts(dropna=False).reset_index()
    freq.columns = [column, "count"]
    freq["percentage"] = round((freq.["count"] / len(df)) * 100, 2)
    freq.to_csv(OUPUT_DIR / f"frequency_{column}.csv", index=False)
    return freq

